# The Security Landscape of FORGE-S Dataset

In [9]:
import os
import json
from dataclasses import dataclass
from dataclasses import field
from typing import List, Dict, Union, Optional, Literal
from pydantic import BaseModel, RootModel, Field
from collections import namedtuple,Counter

@dataclass
class ProjectInfo:
    url: Union[str, int, List, None] = "n/a"
    commit_id: Union[str, int, List, None] = "n/a"
    address: Union[str, int, List, None] = "n/a"
    chain: Union[str, int, List, None] = "n/a"
    compiler_version: Union[str, List, None] = "n/a"
    audit_date: Union[str, int, List, None] = "n/a"
    project_path: Union[str, List, Dict, None] = "n/a"

    def is_empty(self):
        if (self.url == "n/a" and self.address == "n/a") or (
            not self.url and not self.address
        ):
            return True
        return False

@dataclass
class Finding:
    id: Union[str, int] = 0
    category: Dict = field(default_factory=dict)
    title: str = ""
    description: str = ""
    severity: Optional[str] = ""
    location: Union[str, int, List] = ""


class Report(BaseModel):
    path: str = ""
    project_info: ProjectInfo = field(default_factory=ProjectInfo)
    findings: List[Finding] = field(default_factory=list)

    def append_finding(self, finding: Finding):
        self.findings.append(finding)


class JSONFileProcessor:
    def __init__(self, directory: str):
        self.directory = directory
        self.file_count = 0

    def _get_all_json_files(self) -> List[str]:
        json_files = []
        for root, _, files in os.walk(self.directory):
            for file in files:
                if file.endswith(".json"):
                    json_files.append(os.path.join(root, file))
                    self.file_count += 1
        return json_files

    def operate_add(self,results:List,result_type):
        res = {}
        for field in result_type._fields:
            if isinstance(getattr(results[0],field),int):
                res[field] = 0
            else:
                res[field] = []
            # res[field] = 0
        for result in results:
            for field in result._fields:
                res[field] += getattr(result, field)

        return res
    def operate_reduce(self,results:List,result_type):
        res = {}
        for field in result_type._fields:
            res[field] = []
        for result in results:
            for field in result._fields:
                res[field].extend(getattr(result, field))
        return res

    def process_files(
        self, analysis_func=None, modify_func=None
    ) -> List[Report]:
        results = []
        json_files = self._get_all_json_files()
        for json_file in json_files:
            # print(f"Processing file: {json_file}")
            with open(json_file, "r", encoding="utf8") as f:
                data = json.load(f)
            report = Report(**data)
            if analysis_func:
                result = analysis_func(report)
                results.append(result)
            if modify_func:
                modified_report: Report = modify_func(report)
                with open(json_file, "w", encoding="utf8") as f:
                    f.write(modified_report.model_dump_json(indent=4,exclude={'project_info': {'compiler_version'}}))
        return results

class CWE(BaseModel):
    ID: int
    Name: str
    Description: str = ""
    Abstraction: Literal["Pillar", "Class", "Base", "Variant", "Compound"]
    Mapping: Literal["Allowed", "Allowed-with-Review", "Discouraged", "Prohibited"]
    Peer: List = Field(default_factory=list)
    Parent: List = Field(default_factory=list)
    Child: List[int] = Field(default_factory=list)

    def __str__(self) -> str:
        return f"CWE-{self.ID}: {self.Name}"

    def __hash__(self):
        return hash(str(self))

    def add_child(self, child_cwe: "CWE"):
        self.Child.append(child_cwe)
        child_cwe.Parent.append(self)


class CWEDatabase(RootModel):
    root: Dict[str, CWE]

    def get_by_id(self, id: int | str):
        name = f"CWE-{id}"
        return self.root[name]

    def get_by_name(self, name: str):
        return self.root[name]

severity_score_map = {
    "informational": (0, 0, 0),
    "low": (0.1, 3.9, 2),
    "medium": (4.0, 6.9, 5.45),
    "high": (7.0, 8.9, 7.95),
    "critical": (9.0, 10.0, 9.5),
}

class CWEHandler:
    def __init__(self, cwe_database: CWEDatabase):
        self.db = cwe_database
        self.setup_relationships()

    @classmethod
    def load_from_file(cls, file_path: str) -> "CWEHandler":
        import json

        with open(file_path, "r") as f:
            data = json.load(f)
        # CWEDatabase expects a dict where keys are CWE IDs (strings) and values are CWE objects
        db = CWEDatabase.model_validate(data)
        return cls(db)

    def setup_relationships(self):
        """
        Build the parent-child relationships (DAG) based on the 'Child' field (List[int]).
        Populates the 'Parent' field (List[CWE]) for each CWE.
        """
        # First pass: Convert any integer IDs in Parent to CWE objects or clear if invalid
        for cwe in self.db.root.values():
            cleaned_parents = []
            for p in cwe.Parent:
                if isinstance(p, CWE):
                    cleaned_parents.append(p)
                elif isinstance(p, int):
                    try:
                        parent_cwe = self.db.get_by_id(p)
                        cleaned_parents.append(parent_cwe)
                    except KeyError:
                        pass
            cwe.Parent = cleaned_parents

        # Second pass: Build relationships from Child fields
        for cwe_key, cwe in self.db.root.items():
            for child_id in cwe.Child:
                try:
                    child_cwe = self.db.get_by_id(child_id)
                    # Add current cwe as parent to the child
                    # Check if already exists to avoid duplicates if run multiple times
                    if cwe not in child_cwe.Parent:
                        child_cwe.Parent.append(cwe)
                except KeyError:
                    # Child ID might not exist in the database
                    continue

    def get_cwe(self, cwe_id: Union[int, str]) -> Optional[CWE]:
        try:
            if isinstance(cwe_id, int):
                return self.db.get_by_id(cwe_id)
            elif isinstance(cwe_id, str):
                if cwe_id.startswith("CWE-"):
                    return self.db.root[cwe_id]
                else:
                    return self.db.get_by_id(int(cwe_id))
        except (KeyError, ValueError):
            return None

    def get_direct_children(self, cwe_id: Union[int, str]) -> List[CWE]:
        cwe = self.get_cwe(cwe_id)
        if not cwe:
            return []
        children = []
        for child_id in cwe.Child:
            child = self.get_cwe(child_id)
            if child:
                children.append(child)
        return children

    def get_direct_parents(self, cwe_id: Union[int, str]) -> List[CWE]:
        cwe = self.get_cwe(cwe_id)
        if not cwe:
            return []
        return cwe.Parent

    def get_all_descendants(self, cwe_id: Union[int, str]) -> List[CWE]:
        cwe = self.get_cwe(cwe_id)
        if not cwe:
            return []
        descendants = set()
        queue = [cwe]
        while queue:
            current = queue.pop(0)
            for child_id in current.Child:
                child = self.get_cwe(child_id)
                if child and child not in descendants:
                    descendants.add(child)
                    queue.append(child)
        return list(descendants)

    def get_root_parents(self, cwe_id: Union[int, str]) -> List[CWE]:
        """
        Recursively find the most root parents (Pillars).
        """
        cwe = self.get_cwe(cwe_id)
        if not cwe:
            return []

        roots = set()
        queue = [cwe]
        visited = set()

        while queue:
            current = queue.pop(0)
            if current in visited:
                continue
            visited.add(current)

            # If it's a Pillar, it's a root
            if current.Abstraction == "Pillar":
                roots.add(current)
                continue  # Stop traversing up from a Pillar? Usually yes.

            # If no parents, it's a root
            if not current.Parent:
                roots.add(current)
                continue

            # Otherwise, traverse up
            for parent in current.Parent:
                queue.append(parent)

        return list(roots)

    def is_related(self, id1: Union[int, str], id2: Union[int, str]) -> bool:
        """
        Check if there is any inheritance relationship between two CWEs.
        Returns True if id1 is ancestor of id2 OR id2 is ancestor of id1.
        """
        cwe1 = self.get_cwe(id1)
        cwe2 = self.get_cwe(id2)

        if not cwe1 or not cwe2:
            return False

        if cwe1 == cwe2:
            return True

        # Check if cwe1 is ancestor of cwe2
        if self._is_ancestor(cwe1, cwe2):
            return True

        # Check if cwe2 is ancestor of cwe1
        if self._is_ancestor(cwe2, cwe1):
            return True

        return False

    def _is_ancestor(self, ancestor: CWE, descendant: CWE) -> bool:
        # BFS up from descendant to find ancestor
        queue = [descendant]
        visited = set()
        while queue:
            current = queue.pop(0)
            if current in visited:
                continue
            visited.add(current)

            if current == ancestor:
                return True

            for parent in current.Parent:
                queue.append(parent)
        return False


## CWE-Id statistics

In [10]:
CWE = namedtuple("CWE", ["cwe_ids"])

# Load CWE Handler for DAG traversal
try:
    cwe_handler = CWEHandler.load_from_file("../rq5/auditor/cwe_dict.json")
except Exception as e:
    print(f"Warning: Could not load CWE database: {e}")
    cwe_handler = None

def extract_last_cwe_ids(report: Report):
    cwe_ids = []
    for finding in report.findings:
        if finding.category:
            # Get the ID with the highest key from the category dictionary
            try:
                max_key = max([int(k) for k in finding.category.keys() if k.isdigit()])
                cwe_value = finding.category.get(str(max_key), [""])[0]
                # Extract CWE ID if it exists
                if isinstance(cwe_value, str) and cwe_value.startswith("CWE-"):
                    cwe_ids.append(cwe_value)
            except (ValueError, AttributeError):
                continue
    return CWE(cwe_ids=[cwe_id for cwe_id in cwe_ids])

def extract_pillar_cwe_ids(report: Report):
    cwe_ids = []
    for finding in report.findings:
        if finding.category:
            try:
                max_key = max([int(k) for k in finding.category.keys() if k.isdigit()])
                cwe_value = finding.category.get(str(max_key), [""])[0]
                
                if isinstance(cwe_value, str) and cwe_value.startswith("CWE-"):
                    if cwe_handler:
                        # Use CWEHandler to find all Pillar ancestors (DAG support)
                        # e.g., CWE-346 may have multiple Pillar ancestors like CWE-284 and CWE-693
                        pillars = cwe_handler.get_root_parents(cwe_value)
                        
                        for pillar in pillars:
                            cwe_ids.append(f"CWE-{pillar.ID}")
                    else:
                        # Fallback for when handler isn't available: use pre-calculated level 1
                        cwe_value = finding.category.get("1", [""])[0]
                        if isinstance(cwe_value, str) and cwe_value.startswith("CWE-"):
                            cwe_ids.append(cwe_value)
            except (ValueError, AttributeError):
                continue
    return CWE(cwe_ids=[cwe_id for cwe_id in cwe_ids])

processor = JSONFileProcessor("../../dataset/findings")

results = processor.process_files(analysis_func=extract_last_cwe_ids)
results_pillar = processor.process_files(analysis_func=extract_pillar_cwe_ids)


cwe_pillar_result = processor.operate_reduce(results_pillar, CWE)

cwe_result = processor.operate_reduce(results, CWE)

# Count unique CWE IDs
unique_cwe_ids = set(cwe_result["cwe_ids"])
cwe_counter = Counter(cwe_result["cwe_ids"])

print(f"Total unique CWE IDs: {len(unique_cwe_ids)}")

print("Top 10 CWE IDs by occurrence:")
for cwe_id, count in cwe_counter.most_common(10):
    print(f"{cwe_id}: {count}")

print(f"Total unique Pillar CWE IDs: {len(set(cwe_pillar_result['cwe_ids']))}")
print("Top 10 Pillar CWE IDs by occurrence:")
# cwe_pillar_counter_evm = Counter(cwe_pillar_result_evm["cwe_ids"])
cwe_pillar_counter = Counter(cwe_pillar_result["cwe_ids"])
for cwe_id, count in cwe_pillar_counter.most_common(10):
    print(f"{cwe_id}: {count}")


# print(cwe_pillar_counter.most_common(10))
# print(cwe_pillar_counter_evm.most_common(10))

Total unique CWE IDs: 174
Top 10 CWE IDs by occurrence:
CWE-682: 127
CWE-1339: 75
CWE-1284: 74
CWE-1288: 70
CWE-670: 54
CWE-372: 45
CWE-754: 43
CWE-1041: 41
CWE-346: 40
CWE-20: 39
Total unique Pillar CWE IDs: 10
Top 10 Pillar CWE IDs by occurrence:
CWE-682: 259
CWE-664: 255
CWE-707: 247
CWE-710: 221
CWE-693: 146
CWE-691: 140
CWE-284: 119
CWE-703: 110
CWE-435: 52
CWE-697: 33


In [17]:
from collections import Counter, namedtuple

try:
    cwe_handler = CWEHandler.load_from_file("../rq5/evaluator/cwe_dict.json")
    cwe_database = cwe_handler.db # Reference to the underlying database
except Exception as e:
    print(f"Error loading CWE database: {e}")
    # Fallback to manual loading if handler fails (though unlikely if previous cells worked)
    cwe_path = "../rq5/evaluator/cwe_dict.json"
    with open(cwe_path, "r", encoding="utf8") as f:
        cwe_database = CWEDatabase.model_validate_json(f.read())
    cwe_handler = None 


result = namedtuple("CWE", ["cwe_name", "severity_score"])
no_severity_count = {}

def count_cwe(report: Report):
    """Extract CWE IDs and severity scores from findings"""
    findings = report.findings
    cwe_names = []
    severity_score = []
    
    for finding in findings:
        if not finding.category:
            continue

        # Get the CWE from the deepest categorization level
        final_key = max(finding.category.keys())
        cwe_name_list = finding.category.get(final_key, None)
        
        try:
            cwe_list = []
            if cwe_name_list:
                for name in cwe_name_list:
                    # Resolve name to ID
                    try:
                        obj = cwe_database.get_by_name(name)
                        cwe_list.append(obj.ID)
                    except KeyError:
                        pass
            
            cwe_list = cwe_list[0:1]  # Take only the first CWE
        except Exception as e:
            print(f"Error finding {finding.id} in {report.path}: {e}")
            continue
            
        if cwe_list:
            cwe_names.extend(cwe_list)
            
        # Handle severity scoring
        if not finding.severity or finding.severity.lower().strip() == "n/a":
            no_severity_count[cwe_list[0]] = no_severity_count.get(cwe_list[0], 0) + 1
        else:
            # Look up severity score
            s_map = severity_score_map.get(finding.severity.lower().strip(), (0, 0, 0))
            severity_score.append((
                cwe_list[0],
                s_map[2], # Use the score value
            ))
            
    return result(cwe_name=cwe_names, severity_score=severity_score)

# Process all finding files
processor = JSONFileProcessor("../../dataset/findings")
results = processor.process_files(analysis_func=count_cwe)
res = processor.operate_reduce(results, result)

# ---------------- Prepare Basic Statistics ----------------

# Aggregate severity info for weighted average calculation later
cwe_raw_stats = {} # cwe_id -> {'sum_score': float, 'count_rated': int}

for cwe_id, score in res["severity_score"]:
    if cwe_id not in cwe_raw_stats:
        cwe_raw_stats[cwe_id] = {'sum_score': 0.0, 'count_rated': 0}
    cwe_raw_stats[cwe_id]['sum_score'] += score
    cwe_raw_stats[cwe_id]['count_rated'] += 1

# Calculate average severity scores for each CWE (Individual)
cwe_count = Counter(res["cwe_name"])
cwe_severity = {}

for cwe_id, stats in cwe_raw_stats.items():
    if stats['count_rated'] > 0:
        cwe_severity[cwe_id] = stats['sum_score'] / stats['count_rated']
    else:
        cwe_severity[cwe_id] = 0

# Sort by frequency
cwe_count = dict(sorted(cwe_count.items(), key=lambda x: x[1], reverse=True))
cwe_count_backup = cwe_count.copy()
cwe_severity_backup = cwe_severity.copy()



# =============================================================================
# TOP SEVERITY ANALYSIS 
# =============================================================================

print("\nTop 10 CWEs by Average Severity Score:")
original_severity_sorted = sorted(cwe_severity.items(), key=lambda x: x[1], reverse=True)

severity_rank = 0
for cwe_id, severity_score in original_severity_sorted:
    if severity_rank >= 10:
        break
    if cwe_count[cwe_id] > 9:  # Only show CWEs with significant occurrence
        cwe_info = cwe_database.get_by_id(cwe_id)
        print(f"CWE-{cwe_id}: {cwe_info.Name}, {cwe_count[cwe_id]}, {severity_score:.3f}")
        severity_rank += 1


print("\nTop 10 Pillar CWEs by Average Severity Score (DAG-aware):")

pillars = [284, 435, 664, 682, 691, 693, 697, 703, 707, 710]
pillar_hierarchical_stats = []

for pillar_id in pillars:
    if not cwe_handler:
        continue
        
    # 1. Get ALL descendants in the DAG (Unique Set)
    # This handles diamonds (multiple paths to same child) correctly by returning a Set
    descendants = cwe_handler.get_all_descendants(pillar_id)
    
    # 2. Include the pillar itself
    pillar_cwe = cwe_handler.get_cwe(pillar_id)
    relevant_cwes = [pillar_cwe] + descendants if pillar_cwe else descendants
    
    # 3. Aggregate Stats
    total_occurrence = 0
    total_weighted_severity_sum = 0
    total_rated_findings = 0
    cwes_with_severity = 0
    
    for cwe in relevant_cwes:
        cwe_id = cwe.ID
        
        # Count total findings
        count = cwe_count.get(cwe_id, 0)
        total_occurrence += count
        
        # Accumulate severity (Weighted Average strategy)
        # We sum up the raw scores of all underlying findings
        if cwe_id in cwe_raw_stats:
            stats = cwe_raw_stats[cwe_id]
            total_weighted_severity_sum += stats['sum_score']
            total_rated_findings += stats['count_rated']
            if stats['count_rated'] > 0:
                cwes_with_severity += 1
                
    # Calculate Pillar Average
    if total_rated_findings > 0:
        avg_severity = total_weighted_severity_sum / total_rated_findings
    else:
        avg_severity = 0
        
    pillar_hierarchical_stats.append((pillar_id, avg_severity, total_occurrence))

# Sort and Print
pillar_hierarchical_stats.sort(key=lambda x: x[1], reverse=True)

for pillar_id, avg_severity, total_count in pillar_hierarchical_stats:
    if avg_severity > 0:
        cwe_info = cwe_database.get_by_id(pillar_id)
        print(f"CWE-{pillar_id}: {cwe_info.Name}, {total_count} occurrences, {avg_severity:.3f} avg severity")






Top 10 CWEs by Average Severity Score:
CWE-826: Premature Release of Resource During Expected Lifetime, 10, 5.967
CWE-863: Incorrect Authorization, 20, 5.574
CWE-807: Reliance on Untrusted Inputs in a Security Decision, 13, 5.427
CWE-346: Origin Validation Error, 40, 4.922
CWE-666: Operation on Resource in Wrong Phase of Lifetime, 26, 4.886
CWE-682: Incorrect Calculation, 127, 4.806
CWE-345: Insufficient Verification of Data Authenticity, 16, 4.703
CWE-672: Operation on a Resource after Expiration or Release, 10, 4.470
CWE-669: Incorrect Resource Transfer Between Spheres, 13, 4.438
CWE-252: Unchecked Return Value, 16, 4.436

Top 10 Pillar CWEs by Average Severity Score (DAG-aware):
CWE-693: Protection Mechanism Failure, 156 occurrences, 4.299 avg severity
CWE-682: Incorrect Calculation, 259 occurrences, 4.236 avg severity
CWE-284: Improper Access Control, 162 occurrences, 4.197 avg severity
CWE-697: Incorrect Comparison, 34 occurrences, 3.960 avg severity
CWE-691: Insufficient Control

In [18]:
# =============================================================================
# FILTERED ANALYSIS: EXCLUDE INFORMATIONAL FINDINGS (CELL-5 CONSISTENT METHOD)
# =============================================================================
from collections import Counter, namedtuple

filtered_result = namedtuple("CWE", ["cwe_name", "severity_score"])
filtered_no_severity_count = {}

def count_cwe_without_informational(report: Report):
    """Extract CWE IDs and severity scores from findings while excluding informational findings."""
    findings = report.findings
    cwe_names = []
    severity_score = []

    for finding in findings:
        if not finding.category:
            continue

        # Exclude informational findings from all downstream statistics
        severity_text = (finding.severity or "").lower().strip()
        if severity_text == "informational":
            continue

        # Keep the same CWE extraction strategy as cell 5
        final_key = max(finding.category.keys())
        cwe_name_list = finding.category.get(final_key, None)

        try:
            cwe_list = []
            if cwe_name_list:
                for name in cwe_name_list:
                    try:
                        obj = cwe_database.get_by_name(name)
                        cwe_list.append(obj.ID)
                    except KeyError:
                        pass

            cwe_list = cwe_list[0:1]  # Take only the first CWE (same as cell 5)
        except Exception as e:
            print(f"Error finding {finding.id} in {report.path}: {e}")
            continue

        if not cwe_list:
            continue

        cwe_id = cwe_list[0]
        cwe_names.extend(cwe_list)

        # Keep the same severity treatment as cell 5
        if not finding.severity or severity_text == "n/a":
            filtered_no_severity_count[cwe_id] = filtered_no_severity_count.get(cwe_id, 0) + 1
        else:
            s_map = severity_score_map.get(severity_text, (0, 0, 0))
            severity_score.append((cwe_id, s_map[2]))

    return filtered_result(cwe_name=cwe_names, severity_score=severity_score)

# Process all finding files (same pipeline as cell 5)
processor_filtered = JSONFileProcessor("../../dataset/findings")
filtered_results = processor_filtered.process_files(
    analysis_func=count_cwe_without_informational
)
filtered_res = processor_filtered.operate_reduce(filtered_results, filtered_result)

# ---------------- Prepare Basic Statistics (same as cell 5) ----------------
filtered_cwe_raw_stats = {}  # cwe_id -> {'sum_score': float, 'count_rated': int}

for cwe_id, score in filtered_res["severity_score"]:
    if cwe_id not in filtered_cwe_raw_stats:
        filtered_cwe_raw_stats[cwe_id] = {'sum_score': 0.0, 'count_rated': 0}
    filtered_cwe_raw_stats[cwe_id]['sum_score'] += score
    filtered_cwe_raw_stats[cwe_id]['count_rated'] += 1

# Calculate average severity scores for each CWE (Individual)
filtered_cwe_count = Counter(filtered_res["cwe_name"])
filtered_cwe_severity = {}

for cwe_id, stats in filtered_cwe_raw_stats.items():
    if stats['count_rated'] > 0:
        filtered_cwe_severity[cwe_id] = stats['sum_score'] / stats['count_rated']
    else:
        filtered_cwe_severity[cwe_id] = 0

# Sort by frequency (same as cell 5)
filtered_cwe_count = dict(sorted(filtered_cwe_count.items(), key=lambda x: x[1], reverse=True))


print("\nTop 10 CWEs by Average Severity Score:")
filtered_severity_sorted = sorted(filtered_cwe_severity.items(), key=lambda x: x[1], reverse=True)

severity_rank = 0
for cwe_id, severity_score in filtered_severity_sorted:
    if severity_rank >= 10:
        break
    if filtered_cwe_count[cwe_id] > 9:  # Only show CWEs with significant occurrence
        cwe_info = cwe_database.get_by_id(cwe_id)
        print(f"CWE-{cwe_id}: {cwe_info.Name}, {filtered_cwe_count[cwe_id]}, {severity_score:.3f}")
        severity_rank += 1


print("\nTop 10 Pillar CWEs by Average Severity Score (DAG-aware):")

pillars = [284, 435, 664, 682, 691, 693, 697, 703, 707, 710]
pillar_hierarchical_stats = []

for pillar_id in pillars:
    if not cwe_handler:
        continue
        
    # 1. Get ALL descendants in the DAG (Unique Set)
    descendants = cwe_handler.get_all_descendants(pillar_id)
    
    # 2. Include the pillar itself
    pillar_cwe = cwe_handler.get_cwe(pillar_id)
    relevant_cwes = [pillar_cwe] + descendants if pillar_cwe else descendants
    
    # 3. Aggregate Stats
    total_occurrence = 0
    total_weighted_severity_sum = 0
    total_rated_findings = 0
    cwes_with_severity = 0
    
    for cwe in relevant_cwes:
        cwe_id = cwe.ID
        
        # Count total findings
        count = filtered_cwe_count.get(cwe_id, 0)
        total_occurrence += count
        
        # Accumulate severity (Weighted Average strategy)
        if cwe_id in filtered_cwe_raw_stats:
            stats = filtered_cwe_raw_stats[cwe_id]
            total_weighted_severity_sum += stats['sum_score']
            total_rated_findings += stats['count_rated']
            if stats['count_rated'] > 0:
                cwes_with_severity += 1
                
    # Calculate Pillar Average
    if total_rated_findings > 0:
        avg_severity = total_weighted_severity_sum / total_rated_findings
    else:
        avg_severity = 0
        
    pillar_hierarchical_stats.append((pillar_id, avg_severity, total_occurrence))

# Sort and Print (same as cell 5)
pillar_hierarchical_stats.sort(key=lambda x: x[1], reverse=True)

for pillar_id, avg_severity, total_count in pillar_hierarchical_stats:
    if avg_severity > 0:
        cwe_info = cwe_database.get_by_id(pillar_id)
        print(f"CWE-{pillar_id}: {cwe_info.Name}, {total_count} occurrences, {avg_severity:.3f} avg severity")


Top 10 CWEs by Average Severity Score:
CWE-863: Incorrect Authorization, 17, 6.619
CWE-807: Reliance on Untrusted Inputs in a Security Decision, 11, 6.414
CWE-826: Premature Release of Resource During Expected Lifetime, 10, 5.967
CWE-346: Origin Validation Error, 34, 5.791
CWE-252: Unchecked Return Value, 13, 5.645
CWE-1288: Improper Validation of Consistency within Input, 49, 5.570
CWE-682: Incorrect Calculation, 113, 5.407
CWE-345: Insufficient Verification of Data Authenticity, 14, 5.375
CWE-666: Operation on Resource in Wrong Phase of Lifetime, 24, 5.311
CWE-669: Incorrect Resource Transfer Between Spheres, 11, 5.245

Top 10 Pillar CWEs by Average Severity Score (DAG-aware):
CWE-693: Protection Mechanism Failure, 125 occurrences, 5.400 avg severity
CWE-284: Improper Access Control, 130 occurrences, 5.280 avg severity
CWE-682: Incorrect Calculation, 221 occurrences, 4.977 avg severity
CWE-697: Incorrect Comparison, 28 occurrences, 4.809 avg severity
CWE-664: Improper Control of a R

In [ ]:
# =============================================================================
# VISUALIZATION PREPARATION SECTION
# =============================================================================
# The following code prepares data for treemap visualization by flattening
# the CWE hierarchy to better utilize limited display area and show relationships
# the visualization might be distorted due to the DAG nature of CWE inheritance.
# =============================================================================

# Build parent-child relationships for treemap
cwe_count = cwe_count_backup.copy()
cwe_count[284] = 0
cwe_count[707] = 0


def get_parents(cwe_id):
    """Find the first parent CWE that exists in our dataset (For visualization)"""
    try:
        cwe_obj = cwe_database.get_by_id(cwe_id)
        # Parent list is populated by CWEHandler
        params = cwe_obj.Parent
        if not params:
            return ""
        # Try to find a parent that exists in our current dataset
        for p in params:
            # p might be an object or ID depending on handler state, but handler converts to objects
            p_id = p.ID
            if p_id in cwe_count:
                return p_id
        # If no direct parent in dataset, recurse on first parent
        first_p = params[0]
        first_p_id = first_p.ID
        return get_parents(first_p_id)

    except Exception as e:
        print(f"Error finding parent of {cwe_id}: {e}")
        return ""


parents = []
for k, v in cwe_count.items():
    # print(k)
    parent = get_parents(k)
    # print(f"CWE-{k} parent: {parent}")
    if parent != "":
        # parents.append("CWE-" + str(parent))
        parents.append(f"CWE-{parent}")
    else:
        parents.append("")


for k, v in cwe_count.items():
    # print(k)
    parent = get_parents(k)
    # print(f"CWE-{k} parent: {parent}")
    if parent != "":
        # parents.append("CWE-" + str(parent))
        parents.append(f"CWE-{parent}")
    else:
        parents.append("")

# Flatten pillar categories for visualization
# This distributes pillar counts to children to make better use of treemap space
IS_QUANTIFY = False
if IS_QUANTIFY:
    pillars = [284, 435, 664, 682, 691, 693, 697, 703, 707, 710]

    for pillar in pillars:
        children = cwe_database.get_by_id(pillar).Child
        children_counts = [cwe_count.get(child, 0) for child in children]
        pillar_count = cwe_count.get(pillar, 0)

        if pillar_count == 0:
            continue

        total = sum(children_counts)
        if total > 0:
            # Distribute pillar count proportionally to children
            for i, child in enumerate(children):
                if cwe_count.get(child, 0) > 0:
                    cwe_count[child] += int(children_counts[i] / total * pillar_count)

        # Zero out pillar counts after distribution
        cwe_count[pillar] = 0

# Prepare treemap data arrays
values = [v for k, v in cwe_count.items()]
labels = ["CWE-" + str(cwe_database.get_by_id(k).ID) for k, v in cwe_count.items()]

# Initialize colors based on severity scores (squared for better visual contrast)
colors = []
for i in range(len(labels)):
    colors.append(cwe_severity.get(cwe_database.get_by_name(labels[i]).ID, 0) ** 2)

# Add root node for treemap
values.insert(0, 0)
labels.insert(0, "")
colors.insert(0, 0)
parents.insert(0, "")


new_colors = colors.copy()
for i, label in enumerate(labels):
    if label == "":
        continue

    try:
        cwe_id = int(label.split("-")[1])
    except (IndexError, ValueError):
        continue

    # 1. Get all descendants (DAG-aware, handling diamonds)
    relevant_cwes = []
    if cwe_handler:
        # get_all_descendants returns a Unique List/Set of descendant CWE objects

        descendants = cwe_handler.get_all_descendants(cwe_id)
        relevant_cwes = list(descendants)
    else:
        # Fallback if handler is missing (should not happen if cells 8/9 ran)
        try:
            children_ids = cwe_database.get_by_id(cwe_id).Child
            for child_id in children_ids:
                relevant_cwes.append(cwe_database.get_by_id(child_id))
        except:
            pass

    # Add self to the list
    relevant_cwes.append(cwe_database.get_by_id(cwe_id))

    # 2. Aggregate severity stats (Weighted Average)
    total_weighted_severity_sum = 0
    total_rated_findings = 0

    processed_cwe_ids = set()

    for cwe_obj in relevant_cwes:
        if not cwe_obj:
            continue

        cid = cwe_obj.ID
        if cid in processed_cwe_ids:
            continue
        processed_cwe_ids.add(cid)

        if cid in cwe_raw_stats:
            stats = cwe_raw_stats[cid]
            total_weighted_severity_sum += stats["sum_score"]
            total_rated_findings += stats["count_rated"]

    # 3. Calculate Average
    if total_rated_findings > 0:
        avg_severity = total_weighted_severity_sum / total_rated_findings
        # 4. Apply 1.2nd power enhancement (Keep existing visualization characteristic)
        new_colors[i] = avg_severity**1.2
    else:
        new_colors[i] = 0

colors = new_colors


# Normalize colors for visualization (0-1 scale)
max_color = max(colors) if max(colors) > 0 else 1
colors = [(c / max_color) * 2 for c in colors]
colors = [min(c, 1) for c in colors]  # Cap at 1

# Generate titles for hover information
titles = []
for i in range(len(labels)):
    if labels[i] == "":
        titles.append("")
    else:
        titles.append(f"{labels[i]} {cwe_database.get_by_name(labels[i]).Name}")

# Clear annotations for clean visualization when quantified
total = sum(values)
annotations = (
    [str(round(v / total * 1000, 1)) + "‰" for v in values]
    if total > 0
    else ["0.0‰" for v in values]
)

if IS_QUANTIFY:
    for i in range(len(annotations)):
        if annotations[i] == "0.0‰":
            annotations[i] = ""

print("Treemap data prepared:")
print(f"Values (first 10): {values[:10]}")
print(f"Labels (first 10): {labels[:10]}")
print(f"Parents (first 10): {parents[:10]}")

import numpy as np
import plotly.express as px
import plotly.graph_objects as go

palettes = ["#FCF7F3", "#F8EDE3", "#DFD3C3", "#D0B8A8", "#C5705D", "#8D493A"]

colors[0] = 0
fig = go.Figure(
    go.Treemap(
        labels=labels,
        values=values,
        parents=parents,
        maxdepth=2,
        marker=dict(colors=colors, colorscale=palettes),
        # marker=dict(colors=colors, colorscale=px.colors.sequential.aaa),
        # marker=dict(colors=colors, colorscale="amp"),
        textinfo="label+percent entry",
        # texttemplate="%{label}<br>%{text}",
        # texttemplate="%{label}",
        hovertext=titles,
        hoverinfo="text+percent entry",
        textfont=dict(size=10),
    )
)

fig.update_layout(
    margin=dict(t=1, l=1, r=1, b=1),
    # uniformtext=dict(minsize=12),
    # uniformtext=dict(minsize=10,mode="hide"),
    # treemapcolorway = colors,
)
# fig.update_traces(textfont_size=14)
# fig.update_traces(textinfo="label+text")

fig.write_html("cwe_treemap.html")
fig.show()

Treemap data prepared:
Values (first 10): [0, 127, 75, 74, 70, 54, 45, 43, 41, 40]
Labels (first 10): ['', 'CWE-682', 'CWE-1339', 'CWE-1284', 'CWE-1288', 'CWE-670', 'CWE-372', 'CWE-754', 'CWE-1041', 'CWE-346']
Parents (first 10): ['', '', 'CWE-682', 'CWE-20', 'CWE-20', 'CWE-691', 'CWE-664', 'CWE-703', 'CWE-710', 'CWE-284']


## Vulnerabilities Evolution Over Time


In [7]:
import pandas as pd
from collections import namedtuple
import plotly.express as px
import plotly.graph_objects as go

exploit_incidents = {
    "Wormhole Bridge": ("2022-02", 326, 1.0),
    "Cashio Stablecoin": ("2022-03", 52, 0.9),
    "Crema Finance": ("2022-07", 8.78, 1.0),
    "Nirvana Finance": ("2022-07", 3.5, 0.9), 
    "OptiFi": ("2022-08", 0.661, 0.8),
    "Mango Markets": ("2022-10", 110, 1.0),
    "Tulip Protocol": ("2022-10", 2.5, 0.8), 
    "UXD Protocol": ("2022-10", 20, 0.7),    
    "Solend Oracle": ("2022-11", 1.26, 0.65),
    "Raydium DEX": ("2022-12", 5.5, 0.9),
    "Cypher Protocol": ("2023-08", 1, 1.0),
    # "Solflare Wallet Incident": ("2023-11",0,0.9),
    "Loopscale Lending": ("2024-04", 5.8, 1.0),
    "Pump.fun DEX": ("2024-05", 2, 0.9),
    "DEXX Memecoin": ("2024-11", 30, 1.0),
    # "Drift Protocol": ("2026-04", 285, 1.0),

}

CWE = namedtuple("CWE", ["cwe_ids"])


def extract_pillar_cwe_id_dates(report: Report):
    cwe_ids = []
    audit_date = report.project_info.audit_date
    if audit_date == "n/a" or audit_date == None:
        print(f"Invalid audit date in {report.path}")
    for finding in report.findings:
        if finding.category:
            # Get the ID with the highest key from the category dictionary
            try:

                cwe_value = finding.category.get("1", [""])[0]

                # Extract CWE ID if it exists
                if isinstance(cwe_value, str) and cwe_value.startswith("CWE-"):
                    cwe_ids.append((cwe_value, audit_date))
            except (ValueError, AttributeError):
                continue
    return CWE(cwe_ids=[(cwe_id, audit_date) for cwe_id, audit_date in cwe_ids])


processor = JSONFileProcessor("../../dataset/findings")
time_results_pillar = processor.process_files(analysis_func=extract_pillar_cwe_id_dates)
cwe_pillar_time_result = processor.operate_reduce(time_results_pillar, CWE)
cwe_df = pd.DataFrame(cwe_pillar_time_result["cwe_ids"], columns=["cwe_id", "date"])
cwe_df["date"] = pd.to_datetime(cwe_df["date"])

# Group by Quarter (using start_time to keep it as datetime for continuous axis)
cwe_df["quarter"] = cwe_df["date"].dt.to_period("Q").dt.start_time
cwe_df_agg = cwe_df.groupby(["quarter", "cwe_id"]).size().reset_index(name="count")

cwe_df_agg["total_quarterly_count"] = cwe_df_agg.groupby("quarter")["count"].transform(
    "sum"
)
cwe_df_agg["percentage"] = (
    cwe_df_agg["count"] / cwe_df_agg["total_quarterly_count"]
) * 100


cwe_df_agg = cwe_df_agg.sort_values(by="quarter")
# remove the rows with empty quarter
cwe_df_agg = cwe_df_agg.dropna(subset=["quarter"])

# Filter data to include at most 2025-Q3
cwe_df_agg = cwe_df_agg[cwe_df_agg["quarter"] < "2025-10-01"]

exploit_df = pd.DataFrame(
    [
        {"incident": name, "date_str": date, "loss": loss, "y_pos": y_pos}
        for name, (date, loss, y_pos) in exploit_incidents.items()
    ]
)
# Convert to datetime for plotting on the same axis
exploit_df["date"] = pd.to_datetime(exploit_df["date_str"])
exploit_df = exploit_df.sort_values(by=['date', 'loss'], ascending=[True, False])


fig1 = px.area(
    cwe_df_agg,
    x="quarter",
    y="count",
    color="cwe_id",
    line_shape='spline', # Smooth the trend line
    # title="CWE Vulnerabilities Over Time by Pillar",
    labels={
        "quarter": "Quarter",
        "count": "Number of Disclosed Vulnerabilities",
        "cwe_id": "CWE Pillar",
    },
    color_discrete_sequence=px.colors.qualitative.Prism,
)

fig1.update_layout(
    yaxis=dict(        showline=True,
        linewidth=1.4,
        linecolor='black'
        ),
    yaxis2=dict(
        title="Incident Loss (Million $)",
        overlaying="y",
        side="right",
        showgrid=False,
    ),
    plot_bgcolor='white',
    xaxis=dict(
        tickformat="%Y-Q%q", # Format as 2022-Q1
        dtick="M3", # Tick every 3 months
        showline=True,
        linewidth=1.4,
        linecolor='black'
    ),
    xaxis_showgrid=False,
    yaxis_showgrid=False,
    # Add this legend block to precisely position it
    legend=dict(
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.04  # Places the legend's left edge slightly to the right of the plot area
    )
)

# Add exploit incidents as scatter points on the secondary y-axis
for _, row in exploit_df.iterrows():
    fig1.add_trace(
        go.Scatter(
            x=[row["date"]], # Use specific date
            y=[row["loss"]],
            mode="markers",
            # Change marker to a hollow circle (ring)
            marker=dict(symbol="circle-open", size=6, color="indianred", line=dict(width=1.5)),
            name=f"{row['incident']} (${row['loss']}M)",
            yaxis="y2",  # Assign to secondary Y-axis
            hoverinfo="text",
            hovertext=f"{row['incident']}: ${row['loss']}M",
            showlegend=False, # Don't show in legend for individual points
        )
    )
    # Add vertical dashed line
    fig1.add_vline(
        x=row["date"],
        line_width=1,
        line_dash="dash",
        line_color="silver",
        layer="below"
    )
    fig1.add_annotation(
        x=row["date"],
        y=row["y_pos"], # Use the manual y_pos from the DataFrame
        yref="paper",
        text=f"{row['incident']}<br>${row['loss']}M",
        showarrow=False,
        font=dict(size=11, color="grey"),
        xanchor="left",
        yanchor="bottom",
    )



fig2 = px.area(
    cwe_df_agg,
    x="quarter",
    y="percentage",  
    color="cwe_id",
    line_shape='spline', # Smooth the trend line
    # title="Monthly Percentage of CWE Vulnerabilities by Pillar",
    labels={
        "quarter": "Quarter",
        "percentage": "Percentage of Vulnerabilities (%)",
        "cwe_id": "CWE Pillar",
    },
    hover_data={"count": True, "percentage": ":.2f"},
    color_discrete_sequence=px.colors.qualitative.T10,
)
fig1.update_layout(xaxis_title="")
fig2.update_layout(yaxis_ticksuffix="%",xaxis_title="")


fig1.show()
fig2.show()
fig1.write_html("vulnerability_evolution.html")


